In [4]:
import os
import shutil
import glob

# Zmien na swoja nazwe datasetu w /kaggle/input
KAGGLE_DATASET_PATH = "/kaggle/input/datasets/hubertmka/audio-evaluation-mgr"
WORKDIR = "/kaggle/working/Evaluation"

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)

shutil.copytree(KAGGLE_DATASET_PATH, WORKDIR)

print("Skopiowano do:", WORKDIR)
print("Skrypty:")
for p in glob.glob(f"{WORKDIR}/evaluation/evaluate_*.py"):
    print(" -", p)

Skopiowano do: /kaggle/working/Evaluation
Skrypty:
 - /kaggle/working/Evaluation/evaluation/evaluate_sgmse.py
 - /kaggle/working/Evaluation/evaluation/evaluate_melregan.py
 - /kaggle/working/Evaluation/evaluation/evaluate_cdiffuse.py
 - /kaggle/working/Evaluation/evaluation/evaluate_metricgan.py
 - /kaggle/working/Evaluation/evaluation/evaluate_convtasnet.py
 - /kaggle/working/Evaluation/evaluation/evaluate_hifigan.py
 - /kaggle/working/Evaluation/evaluation/evaluate_cleanunet.py


In [ ]:
%cd /kaggle/working/Evaluation

!python -m pip install \
  numpy scipy scikit-learn \
  librosa soundfile tqdm \
  speechbrain asteroid torchmetrics \
  pyroomacoustics sdeint \
  torch torchaudio torch_ema asteroid-filterbanks

In [ ]:
!pip uninstall -y asteroid

In [ ]:
!pip install -q -U "torchmetrics[audio]==1.9.0" onnxruntime

In [5]:
%cd /kaggle/working/Evaluation
!mkdir -p /kaggle/working/Evaluation/blind_testset/noisy
!find /kaggle/working/Evaluation/blind_testset -maxdepth 1 -name "*.wav" -exec cp {} /kaggle/working/Evaluation/blind_testset/noisy/ \;

/kaggle/working/Evaluation


In [6]:
%cd /kaggle/working/Evaluation

!python evaluation/evaluate_sgmse.py \
  --root_dir /kaggle/working/Evaluation \
  --subset blind_testset \
  --enhanced_dir /kaggle/working/results/sgmse-blind \
  --ckpt /kaggle/working/Evaluation/checkpoints/sgmse/train_vb_29nqe0uh_epoch=115.ckpt \
  --device cuda

/kaggle/working/Evaluation
Found 743 noisy files.
Selected 743 files (start_idx=0, end_idx=743).

[Enhancement] Running command:
/usr/bin/python3 /kaggle/working/Evaluation/sgmse/enhancement.py --test_dir /kaggle/working/Evaluation/blind_testset/noisy --enhanced_dir /kaggle/working/results/sgmse-blind --ckpt /kaggle/working/Evaluation/checkpoints/sgmse/train_vb_29nqe0uh_epoch=115.ckpt --sampler_type pc --corrector ald --corrector_steps 1 --snr 0.5 --N 30 --device cuda --t_eps 0.03 --metadata_csv /kaggle/working/results/sgmse-blind/_enhancement_manifest.csv
Set TORCH_CUDA_ARCH_LIST to: 7.5;7.5
Lightning automatically upgraded your loaded checkpoint from v1.5.10 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint checkpoints/sgmse/train_vb_29nqe0uh_epoch=115.ckpt`
100%|███████████████████████████████████████| 743/743 [9:20:40<00:00, 45.28s/it]
Saved enhanced files to: /kaggle/working/results/sgmse-blind
Saved manifest: 

In [7]:
%cd /kaggle/working/Evaluation
!python evaluation/evaluate_cdiffuse.py \
  --root_dir . \
  --subset /kaggle/working/Evaluation/blind_testset \
  --enhanced_dir /kaggle/working/results/cdiffuse-blind \
  --ckpt ./checkpoints/cdiffuse/weights-370200.pt \
  --cdiffuse_dir ./CDiffuSE \
  --device cuda

/kaggle/working/Evaluation
Found 743 noisy files.
Selected 743 files (start_idx=0, end_idx=743).

[CDiffuSE Preprocess] Running command:
/usr/bin/python3 preprocess.py /kaggle/working/results/cdiffuse-blind/_cdiffuse_tmp/noisy_flat /kaggle/working/results/cdiffuse-blind/_cdiffuse_tmp/spec_flat --se --test --voicebank
Preprocessing: 100%|██████████████████████████| 743/743 [00:47<00:00, 15.55it/s]

[CDiffuSE Inference] Running command:
/usr/bin/python3 inference.py /kaggle/working/Evaluation/checkpoints/cdiffuse/weights-370200.pt /kaggle/working/results/cdiffuse-blind/_cdiffuse_tmp/spec_flat /kaggle/working/results/cdiffuse-blind/_cdiffuse_tmp/noisy_flat -o /kaggle/working/results/cdiffuse-blind --se --voicebank --device cuda --metadata_csv /kaggle/working/results/cdiffuse-blind/_cdiffuse_tmp/_inference_metadata_raw.csv --fast
2026-05-10 00:45:43.851506: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for pl

In [8]:
%cd /kaggle/working/Evaluation
!python evaluation/evaluate_cleanunet.py \
  --root_dir . \
  --subset /kaggle/working/Evaluation/blind_testset \
  --enhanced_dir /kaggle/working/results/cleanunet-blind \
  --cleanunet_dir ./CleanUNet \
  --config configs/DNS-large-high.json \
  --ckpt_iter pretrained \
  --device cuda

/kaggle/working/Evaluation
Found 743 noisy files.
Selected 743 files (start_idx=0, end_idx=743).
/kaggle/working/Evaluation/CleanUNet/util.py:188: SyntaxWarning: invalid escape sequence '\e'
  ell_p: \ell_p norm (1 or 2) of the AE loss
CleanUNet enhancement: 100%|██████████████████| 743/743 [01:12<00:00, 10.20it/s]
Saved enhanced files to: /kaggle/working/results/cleanunet-blind
Saved manifest: /kaggle/working/results/cleanunet-blind/_enhancement_manifest.csv


In [9]:
%cd /kaggle/working/Evaluation
!python evaluation/evaluate_convtasnet.py \
  --root_dir . \
  --subset /kaggle/working/Evaluation/blind_testset \
  --enhanced_dir /kaggle/working/results/convtasnet-blind \
  --hub_weights JorisCos/ConvTasNet_Libri1Mix_enhsingle_16k \
  --device cuda

/kaggle/working/Evaluation
Found 743 noisy files.
Selected 743 files (start_idx=0, end_idx=743).
/usr/local/lib/python3.12/dist-packages/torch/hub.py:247: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to load(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  _check_repo_is_trusted(
Downloading: "https://github.com/mpariente/asteroid/zipball/master" to /root/.cache/torch/hub/master.zip
/root/.cache/torch/hub/mpariente_asteroid_master/asteroid/masknn/tac.py:42: SyntaxWarning: invalid escape sequence '\_'
  Shape: :math:`(batch, mic\_ch

In [10]:
%cd /kaggle/working/Evaluation
!python evaluation/evaluate_metricgan.py \
  --root_dir . \
  --subset /kaggle/working/Evaluation/blind_testset \
  --enhanced_dir /kaggle/working/results/metricgan-blind \
  --savedir ./pretrained_models/metricgan-plus-voicebank \
  --device cuda

/kaggle/working/Evaluation
Found 743 noisy files.
Selected 743 files (start_idx=0, end_idx=743).
Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.
MetricGAN+ enhancement: 100%|█████████████████| 743/743 [00:28<00:00, 26.13it/s]
Saved enhanced files to: /kaggle/working/results/metricgan-blind
Saved manifest: /kaggle/working/results/metricgan-blind/_enhancement_manifest.csv


In [11]:
%cd /kaggle/working/Evaluation
!python evaluation/evaluate_hifigan.py \
  --root_dir . \
  --subset /kaggle/working/Evaluation/blind_testset \
  --enhanced_dir /kaggle/working/results/hifigan-blind \
  --hifigan_savedir ./pretrained_models/speechbrain-hifigan-16k \
  --device cuda

/kaggle/working/Evaluation
Found 743 noisy files.
Selected 743 files (start_idx=0, end_idx=743).
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.
HiFiGAN baseline: 100%|███████████████████████| 743/743 [02:19<00:00,  5.34it/s]
Saved enhanced files to: /kaggle/working/results/hifigan-blind
Saved manifest: /kaggle/working/results/hifigan-blind/_enhancement_manifest.csv


In [12]:
%cd /kaggle/working/Evaluation
!python evaluation/evaluate_melregan.py \
  --root_dir . \
  --subset /kaggle/working/Evaluation/blind_testset \
  --enhanced_dir /kaggle/working/results/melregan-blind \
  --ckpt ./checkpoints/melregan/MelReGAN-epoch=50-val_loss=28.4744.ckpt \
  --hifigan_savedir ./pretrained_models/speechbrain-hifigan-16k \
  --device cuda

/kaggle/working/Evaluation
Found 743 noisy files.
Selected 743 files (start_idx=0, end_idx=743).
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.
MelReGAN enhancement: 100%|███████████████████| 743/743 [02:50<00:00,  4.35it/s]
Saved enhanced files to: /kaggle/working/results/melregan-blind
Saved manifest: /kaggle/working/results/melregan-blind/_enhancement_manifest.csv


In [ ]:
!python /kaggle/working/Evaluation/evaluation/calculate_metrics.py --noisy_dir /kaggle/working/Evaluation/test/noisy --clean_dir /kaggle/working/Evaluation/test/clean_aligned --enhanced_dir /kaggle/working/results/melregan-100ms --output_csv /kaggle/working/results/melregan_metrics-100ms.csv --device cpu --skip_wer

In [11]:
!ls /kaggle/working/Evaluation/evaluation

calculate_metrics_blind.py  evaluate_cleanunet.py   evaluate_melregan.py
calculate_metrics.py	    evaluate_convtasnet.py  evaluate_metricgan.py
evaluate_cdiffuse.py	    evaluate_hifigan.py     evaluate_sgmse.py


In [10]:
!cp /kaggle/input/datasets/hubertmka/blind-metrics/calculate_metrics_blind.py /kaggle/working/Evaluation/evaluation/calculate_metrics_blind.py

In [17]:
!ls /kaggle/working/Evaluation/blind_testset

00000.wav  00107.wav  00214.wav  00321.wav  00428.wav  00535.wav  00642.wav
00001.wav  00108.wav  00215.wav  00322.wav  00429.wav  00536.wav  00643.wav
00002.wav  00109.wav  00216.wav  00323.wav  00430.wav  00537.wav  00644.wav
00003.wav  00110.wav  00217.wav  00324.wav  00431.wav  00538.wav  00645.wav
00004.wav  00111.wav  00218.wav  00325.wav  00432.wav  00539.wav  00646.wav
00005.wav  00112.wav  00219.wav  00326.wav  00433.wav  00540.wav  00647.wav
00006.wav  00113.wav  00220.wav  00327.wav  00434.wav  00541.wav  00648.wav
00007.wav  00114.wav  00221.wav  00328.wav  00435.wav  00542.wav  00649.wav
00008.wav  00115.wav  00222.wav  00329.wav  00436.wav  00543.wav  00650.wav
00009.wav  00116.wav  00223.wav  00330.wav  00437.wav  00544.wav  00651.wav
00010.wav  00117.wav  00224.wav  00331.wav  00438.wav  00545.wav  00652.wav
00011.wav  00118.wav  00225.wav  00332.wav  00439.wav  00546.wav  00653.wav
00012.wav  00119.wav  00226.wav  00333.wav  00440.wav  00547.wav  00654.wav
00013.wav  0

In [20]:
%cd /kaggle/working/Evaluation

!python evaluation/calculate_metrics_blind.py \
  --blind_dir blind_testset \
  --output_csv /kaggle/working/results/blind_metrics.csv \
  --device cuda

/kaggle/working/Evaluation
[info] Indexing blind files...
[info] Blind files: 743
[info] [1/743] 00000.wav
[info] [2/743] 00001.wav
[info] [3/743] 00002.wav
[info] [4/743] 00003.wav
[info] [5/743] 00004.wav
[info] [6/743] 00005.wav
[info] [7/743] 00006.wav
[info] [8/743] 00007.wav
[info] [9/743] 00008.wav
[info] [10/743] 00009.wav
[info] [11/743] 00010.wav
[info] [12/743] 00011.wav
[info] [13/743] 00012.wav
[info] [14/743] 00013.wav
[info] [15/743] 00014.wav
[info] [16/743] 00015.wav
[info] [17/743] 00016.wav
[info] [18/743] 00017.wav
[info] [19/743] 00018.wav
[info] [20/743] 00019.wav
[info] [21/743] 00020.wav
[info] [22/743] 00021.wav
[info] [23/743] 00022.wav
[info] [24/743] 00023.wav
[info] [25/743] 00024.wav
[info] [26/743] 00025.wav
[info] [27/743] 00026.wav
[info] [28/743] 00027.wav
[info] [29/743] 00028.wav
[info] [30/743] 00029.wav
[info] [31/743] 00030.wav
[info] [32/743] 00031.wav
[info] [33/743] 00032.wav
[info] [34/743] 00033.wav
[info] [35/743] 00034.wav
[info] [36/743] 0

In [21]:
%cd /kaggle/working/Evaluation

!python evaluation/calculate_metrics_blind.py \
  --blind_dir /kaggle/working/results/cdiffuse-blind \
  --output_csv /kaggle/working/results/blind_cdiffuse_metrics.csv \
  --device cuda

/kaggle/working/Evaluation
[info] Indexing blind files...
[info] Blind files: 743
[info] [1/743] 00000.wav
[info] [2/743] 00001.wav
[info] [3/743] 00002.wav
[info] [4/743] 00003.wav
[info] [5/743] 00004.wav
[info] [6/743] 00005.wav
[info] [7/743] 00006.wav
[info] [8/743] 00007.wav
[info] [9/743] 00008.wav
[info] [10/743] 00009.wav
[info] [11/743] 00010.wav
[info] [12/743] 00011.wav
[info] [13/743] 00012.wav
[info] [14/743] 00013.wav
[info] [15/743] 00014.wav
[info] [16/743] 00015.wav
[info] [17/743] 00016.wav
[info] [18/743] 00017.wav
[info] [19/743] 00018.wav
[info] [20/743] 00019.wav
[info] [21/743] 00020.wav
[info] [22/743] 00021.wav
[info] [23/743] 00022.wav
[info] [24/743] 00023.wav
[info] [25/743] 00024.wav
[info] [26/743] 00025.wav
[info] [27/743] 00026.wav
[info] [28/743] 00027.wav
[info] [29/743] 00028.wav
[info] [30/743] 00029.wav
[info] [31/743] 00030.wav
[info] [32/743] 00031.wav
[info] [33/743] 00032.wav
[info] [34/743] 00033.wav
[info] [35/743] 00034.wav
[info] [36/743] 0

In [22]:
%cd /kaggle/working/Evaluation

!python evaluation/calculate_metrics_blind.py \
  --blind_dir /kaggle/working/results/cleanunet-blind \
  --output_csv /kaggle/working/results/blind_cleanunet_metrics.csv \
  --device cuda

/kaggle/working/Evaluation
[info] Indexing blind files...
[info] Blind files: 743
[info] [1/743] 00000.wav
[info] [2/743] 00001.wav
[info] [3/743] 00002.wav
[info] [4/743] 00003.wav
[info] [5/743] 00004.wav
[info] [6/743] 00005.wav
[info] [7/743] 00006.wav
[info] [8/743] 00007.wav
[info] [9/743] 00008.wav
[info] [10/743] 00009.wav
[info] [11/743] 00010.wav
[info] [12/743] 00011.wav
[info] [13/743] 00012.wav
[info] [14/743] 00013.wav
[info] [15/743] 00014.wav
[info] [16/743] 00015.wav
[info] [17/743] 00016.wav
[info] [18/743] 00017.wav
[info] [19/743] 00018.wav
[info] [20/743] 00019.wav
[info] [21/743] 00020.wav
[info] [22/743] 00021.wav
[info] [23/743] 00022.wav
[info] [24/743] 00023.wav
[info] [25/743] 00024.wav
[info] [26/743] 00025.wav
[info] [27/743] 00026.wav
[info] [28/743] 00027.wav
[info] [29/743] 00028.wav
[info] [30/743] 00029.wav
[info] [31/743] 00030.wav
[info] [32/743] 00031.wav
[info] [33/743] 00032.wav
[info] [34/743] 00033.wav
[info] [35/743] 00034.wav
[info] [36/743] 0

In [23]:
%cd /kaggle/working/Evaluation

!python evaluation/calculate_metrics_blind.py \
  --blind_dir /kaggle/working/results/convtasnet-blind \
  --output_csv /kaggle/working/results/blind_convtasnet_metrics.csv \
  --device cuda

/kaggle/working/Evaluation
[info] Indexing blind files...
[info] Blind files: 743
[info] [1/743] 00000.wav
[info] [2/743] 00001.wav
[info] [3/743] 00002.wav
[info] [4/743] 00003.wav
[info] [5/743] 00004.wav
[info] [6/743] 00005.wav
[info] [7/743] 00006.wav
[info] [8/743] 00007.wav
[info] [9/743] 00008.wav
[info] [10/743] 00009.wav
[info] [11/743] 00010.wav
[info] [12/743] 00011.wav
[info] [13/743] 00012.wav
[info] [14/743] 00013.wav
[info] [15/743] 00014.wav
[info] [16/743] 00015.wav
[info] [17/743] 00016.wav
[info] [18/743] 00017.wav
[info] [19/743] 00018.wav
[info] [20/743] 00019.wav
[info] [21/743] 00020.wav
[info] [22/743] 00021.wav
[info] [23/743] 00022.wav
[info] [24/743] 00023.wav
[info] [25/743] 00024.wav
[info] [26/743] 00025.wav
[info] [27/743] 00026.wav
[info] [28/743] 00027.wav
[info] [29/743] 00028.wav
[info] [30/743] 00029.wav
[info] [31/743] 00030.wav
[info] [32/743] 00031.wav
[info] [33/743] 00032.wav
[info] [34/743] 00033.wav
[info] [35/743] 00034.wav
[info] [36/743] 0

In [24]:
%cd /kaggle/working/Evaluation

!python evaluation/calculate_metrics_blind.py \
  --blind_dir /kaggle/working/results/hifigan-blind \
  --output_csv /kaggle/working/results/blind_hifigan_metrics.csv \
  --device cuda

/kaggle/working/Evaluation
[info] Indexing blind files...
[info] Blind files: 743
[info] [1/743] 00000.wav
[info] [2/743] 00001.wav
[info] [3/743] 00002.wav
[info] [4/743] 00003.wav
[info] [5/743] 00004.wav
[info] [6/743] 00005.wav
[info] [7/743] 00006.wav
[info] [8/743] 00007.wav
[info] [9/743] 00008.wav
[info] [10/743] 00009.wav
[info] [11/743] 00010.wav
[info] [12/743] 00011.wav
[info] [13/743] 00012.wav
[info] [14/743] 00013.wav
[info] [15/743] 00014.wav
[info] [16/743] 00015.wav
[info] [17/743] 00016.wav
[info] [18/743] 00017.wav
[info] [19/743] 00018.wav
[info] [20/743] 00019.wav
[info] [21/743] 00020.wav
[info] [22/743] 00021.wav
[info] [23/743] 00022.wav
[info] [24/743] 00023.wav
[info] [25/743] 00024.wav
[info] [26/743] 00025.wav
[info] [27/743] 00026.wav
[info] [28/743] 00027.wav
[info] [29/743] 00028.wav
[info] [30/743] 00029.wav
[info] [31/743] 00030.wav
[info] [32/743] 00031.wav
[info] [33/743] 00032.wav
[info] [34/743] 00033.wav
[info] [35/743] 00034.wav
[info] [36/743] 0

In [25]:
%cd /kaggle/working/Evaluation

!python evaluation/calculate_metrics_blind.py \
  --blind_dir /kaggle/working/results/melregan-blind \
  --output_csv /kaggle/working/results/blind_melregan_metrics.csv \
  --device cuda

/kaggle/working/Evaluation
[info] Indexing blind files...
[info] Blind files: 743
[info] [1/743] 00000.wav
[info] [2/743] 00001.wav
[info] [3/743] 00002.wav
[info] [4/743] 00003.wav
[info] [5/743] 00004.wav
[info] [6/743] 00005.wav
[info] [7/743] 00006.wav
[info] [8/743] 00007.wav
[info] [9/743] 00008.wav
[info] [10/743] 00009.wav
[info] [11/743] 00010.wav
[info] [12/743] 00011.wav
[info] [13/743] 00012.wav
[info] [14/743] 00013.wav
[info] [15/743] 00014.wav
[info] [16/743] 00015.wav
[info] [17/743] 00016.wav
[info] [18/743] 00017.wav
[info] [19/743] 00018.wav
[info] [20/743] 00019.wav
[info] [21/743] 00020.wav
[info] [22/743] 00021.wav
[info] [23/743] 00022.wav
[info] [24/743] 00023.wav
[info] [25/743] 00024.wav
[info] [26/743] 00025.wav
[info] [27/743] 00026.wav
[info] [28/743] 00027.wav
[info] [29/743] 00028.wav
[info] [30/743] 00029.wav
[info] [31/743] 00030.wav
[info] [32/743] 00031.wav
[info] [33/743] 00032.wav
[info] [34/743] 00033.wav
[info] [35/743] 00034.wav
[info] [36/743] 0

In [26]:
%cd /kaggle/working/Evaluation

!python evaluation/calculate_metrics_blind.py \
  --blind_dir /kaggle/working/results/metricgan-blind \
  --output_csv /kaggle/working/results/blind_metricgan_metrics.csv \
  --device cuda

/kaggle/working/Evaluation
[info] Indexing blind files...
[info] Blind files: 743
[info] [1/743] 00000.wav
[info] [2/743] 00001.wav
[info] [3/743] 00002.wav
[info] [4/743] 00003.wav
[info] [5/743] 00004.wav
[info] [6/743] 00005.wav
[info] [7/743] 00006.wav
[info] [8/743] 00007.wav
[info] [9/743] 00008.wav
[info] [10/743] 00009.wav
[info] [11/743] 00010.wav
[info] [12/743] 00011.wav
[info] [13/743] 00012.wav
[info] [14/743] 00013.wav
[info] [15/743] 00014.wav
[info] [16/743] 00015.wav
[info] [17/743] 00016.wav
[info] [18/743] 00017.wav
[info] [19/743] 00018.wav
[info] [20/743] 00019.wav
[info] [21/743] 00020.wav
[info] [22/743] 00021.wav
[info] [23/743] 00022.wav
[info] [24/743] 00023.wav
[info] [25/743] 00024.wav
[info] [26/743] 00025.wav
[info] [27/743] 00026.wav
[info] [28/743] 00027.wav
[info] [29/743] 00028.wav
[info] [30/743] 00029.wav
[info] [31/743] 00030.wav
[info] [32/743] 00031.wav
[info] [33/743] 00032.wav
[info] [34/743] 00033.wav
[info] [35/743] 00034.wav
[info] [36/743] 0

In [27]:
%cd /kaggle/working/Evaluation

!python evaluation/calculate_metrics_blind.py \
  --blind_dir /kaggle/working/results/sgmse-blind \
  --output_csv /kaggle/working/results/blind_sgmse_metrics.csv \
  --device cuda

/kaggle/working/Evaluation
[info] Indexing blind files...
[info] Blind files: 743
[info] [1/743] 00000.wav
[info] [2/743] 00001.wav
[info] [3/743] 00002.wav
[info] [4/743] 00003.wav
[info] [5/743] 00004.wav
[info] [6/743] 00005.wav
[info] [7/743] 00006.wav
[info] [8/743] 00007.wav
[info] [9/743] 00008.wav
[info] [10/743] 00009.wav
[info] [11/743] 00010.wav
[info] [12/743] 00011.wav
[info] [13/743] 00012.wav
[info] [14/743] 00013.wav
[info] [15/743] 00014.wav
[info] [16/743] 00015.wav
[info] [17/743] 00016.wav
[info] [18/743] 00017.wav
[info] [19/743] 00018.wav
[info] [20/743] 00019.wav
[info] [21/743] 00020.wav
[info] [22/743] 00021.wav
[info] [23/743] 00022.wav
[info] [24/743] 00023.wav
[info] [25/743] 00024.wav
[info] [26/743] 00025.wav
[info] [27/743] 00026.wav
[info] [28/743] 00027.wav
[info] [29/743] 00028.wav
[info] [30/743] 00029.wav
[info] [31/743] 00030.wav
[info] [32/743] 00031.wav
[info] [33/743] 00032.wav
[info] [34/743] 00033.wav
[info] [35/743] 00034.wav
[info] [36/743] 0

In [2]:
!cp /kaggle/input/datasets/hubertmka/squim-script/patch_squim_mos_cloud.py /kaggle/working/Evaluation/evaluation/patch_squim_mos_cloud.py

In [18]:
%%bash
SCRIPT=/kaggle/working/Evaluation/evaluation/patch_squim_mos_cloud.py
CSV_IN=/kaggle/working/results
NOISY=/kaggle/input/datasets/hubertmka/audio-evaluation-mgr/blind_testset
NMR=/kaggle/input/datasets/hubertmka/clean-aligned/test/clean_aligned
OUT=/kaggle/working/MGR_Wyniki
mkdir -p $OUT

for M in cdiffuse cleanunet convtasnet hifigan metricgan sgmse; do
  echo "=== $M ==="
  python -u $SCRIPT \
    --csv          $CSV_IN/blind_${M}_metrics.csv \
    --enhanced_dir /kaggle/working/results/${M}-blind \
    --noisy_dir    $NOISY \
    --nmr_dir      $NMR \
    --output_csv   $OUT/blind_${M}_metrics.csv \
    --device cuda
done
ls -la $OUT

=== cdiffuse ===
[info] loading SQUIM_SUBJECTIVE on cuda...
[info] NMR: /kaggle/input/datasets/hubertmka/clean-aligned/test/clean_aligned/p107/00412_-0.7dB.wav
[info] squim_mos: 743 rows to compute (from /kaggle/working/results/cdiffuse-blind)
  [50/743] 00049.wav squim_mos=2.442
  [100/743] 00099.wav squim_mos=3.787
  [150/743] 00149.wav squim_mos=3.996
  [200/743] 00199.wav squim_mos=2.213
  [250/743] 00249.wav squim_mos=2.657
  [300/743] 00299.wav squim_mos=3.222
  [350/743] 00349.wav squim_mos=2.213
  [400/743] 00399.wav squim_mos=3.692
  [450/743] 00449.wav squim_mos=2.929
  [500/743] 00499.wav squim_mos=2.478
  [550/743] 00549.wav squim_mos=2.995
  [600/743] 00599.wav squim_mos=3.096
  [650/743] 00649.wav squim_mos=2.197
  [700/743] 00699.wav squim_mos=3.115
  [743/743] 00742.wav squim_mos=2.220
[info] noisy_squim_mos: 743 rows to compute (from /kaggle/input/datasets/hubertmka/audio-evaluation-mgr/blind_testset)
  [50/743] 00049.wav noisy_squim_mos=2.363
  [100/743] 00099.wav noi

In [17]:
!ls /kaggle/input/
!find /kaggle/input -maxdepth 3 -type d -iname "*clean*"

datasets
/kaggle/input/datasets/hubertmka/clean-aligned
